# FreshCheck-Agent v2: entrenamiento en Google Colab (GPU)

Ejecuta el ajuste fino del Hito 3 en una GPU gratuita. **Antes de empezar:** Entorno de ejecución → Cambiar tipo → GPU (T4).

Necesita subir a su Google Drive el archivo `Original Image.zip` de FruitVision (https://data.mendeley.com/datasets/xkbjx8959c/2; licencia CC BY-NC-ND 4.0, uso académico, no redistribuir).

In [1]:

# Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# Ruta de la carpeta Original Image
ORIGEN = '/content/drive/MyDrive/FreshCheck/Original Image'

# Comprobar que existe
assert os.path.isdir(ORIGEN), (
    f'No se encontró la carpeta: {ORIGEN}. '
    'Comprueba su ubicación en Google Drive.'
)

print('Dataset encontrado correctamente.')
print('Ubicación:', ORIGEN)
print('Contenido:', os.listdir(ORIGEN)[:10])


Mounted at /content/drive
Dataset encontrado correctamente.
Ubicación: /content/drive/MyDrive/FreshCheck/Original Image
Contenido: ['Fruits Original']


In [2]:
!git clone https://github.com/PedroC1213/FreshCheck-Agent.git
%cd FreshCheck-Agent
!git log --oneline -3

Cloning into 'FreshCheck-Agent'...
remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (272/272), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 272 (delta 97), reused 196 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (272/272), 6.73 MiB | 16.61 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/FreshCheck-Agent
c9ad7a8 (HEAD -> main, origin/main, origin/HEAD) Corregir el cuaderno de Colab (JSON valido)
bb6ef63 Set RAPIDO to True for faster training
798bc93 Modo  para el Hito 3


Colab ya trae torch, numpy, pandas y scikit-learn. Se usan esas versiones (quedan registradas en `entorno.json`); solo se agregan las faltantes.

In [2]:


# ============================================================
# BUSCAR ORIGINAL IMAGE EN GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive'

print('CARPETAS EN MI UNIDAD:\n')

for elemento in os.listdir(BASE):
    ruta = os.path.join(BASE, elemento)
    tipo = 'CARPETA' if os.path.isdir(ruta) else 'ARCHIVO'
    print(tipo, '->', elemento)

print('\nBUSCANDO FRESHCHECK...')

for raiz, carpetas, archivos in os.walk(BASE):
    if 'FreshCheck' in carpetas:
        print('FreshCheck encontrada:', os.path.join(raiz, 'FreshCheck'))

    if 'Original Image' in carpetas:
        print('Original Image encontrada:', os.path.join(raiz, 'Original Image'))

    for archivo in archivos:
        if archivo.lower() == 'original image.zip':
            print('ZIP encontrado:', os.path.join(raiz, archivo))



Mounted at /content/drive
CARPETAS EN MI UNIDAD:

ARCHIVO -> comprobante pago(1) (1).pdf
ARCHIVO -> cédula.(2) (1).pdf
ARCHIVO -> comprobante de pago .pdf
ARCHIVO -> Pago Certificado de estudio.pdf
ARCHIVO -> pago certificado de notas (1).pdf
ARCHIVO -> pago certificado de notas.pdf
ARCHIVO -> pago certificado.pdf
ARCHIVO -> Pago certificado estudió.pdf
ARCHIVO -> CamScanner 21-06-2023 10.16.pdf
ARCHIVO -> DOC-20230621-WA0000..pdf
ARCHIVO -> RECIBO.pdf
ARCHIVO -> cc (4).pdf
ARCHIVO -> cc (3).pdf
ARCHIVO -> cc (2).pdf
ARCHIVO -> cc (1).pdf
ARCHIVO -> cc.pdf
ARCHIVO -> constancia de pago .pdf
ARCHIVO -> DOC-20240219-WA0007..pdf
ARCHIVO -> API de Drag and drop - Word 2024-04-07 14-26-07.mp4
CARPETA -> cosas pedro
ARCHIVO -> 240. PEDRO JOSE CASTRO BLANQUICETT(2).pdf
ARCHIVO -> promedio.pdf
ARCHIVO -> desprendible pago.pdf
ARCHIVO -> DOC-20240702-WA0006..pdf
ARCHIVO -> Buen desempeño (2).pdf
ARCHIVO -> Buen desempeño (1).pdf
ARCHIVO -> Buen desempeño.pdf
ARCHIVO -> DOC-20240708-WA0004.

Se usan el `data/manifest.csv` y `data/splits.csv` **del repositorio** (partición idéntica a la del Hito 2). Compruebe que el hash coincide:

In [4]:


# ============================================================
# CLONAR Y CONFIGURAR FRESHCHECK-AGENT
# ============================================================

import os
import subprocess
from pathlib import Path

REPO = '/content/FreshCheck-Agent'

# Comprobar si el repositorio ya existe
if not os.path.isdir(os.path.join(REPO, '.git')):

    print('Clonando FreshCheck-Agent desde GitHub...')

    # Si existe una carpeta incompleta, eliminarla
    if os.path.exists(REPO):
        import shutil
        shutil.rmtree(REPO)

    subprocess.run(
        [
            'git',
            'clone',
            'https://github.com/PedroC1213/FreshCheck-Agent.git',
            REPO
        ],
        check=True
    )

else:
    print('Repositorio encontrado.')

# Entrar al proyecto
os.chdir(REPO)

print('\nDirectorio actual:')
print(os.getcwd())

# Mostrar últimos commits
print('\nÚltimos commits:')
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

# Comprobar carpetas principales
print('\nContenido del repositorio:')
print(os.listdir(REPO))

print('\nREPOSITORIO CONFIGURADO CORRECTAMENTE.')


Clonando FreshCheck-Agent desde GitHub...

Directorio actual:
/content/FreshCheck-Agent

Últimos commits:

Contenido del repositorio:
['models_v2', 'app.py', 'data', 'README.md', 'logs', 'scripts', 'analysis', 'freshcheck', 'experiments', 'models', 'config', 'requirements.txt', '.git', 'tests', 'src', '.gitignore', 'requirements-v1.txt', 'docs', 'app_v2.py', 'notebooks']

REPOSITORIO CONFIGURADO CORRECTAMENTE.


In [5]:

# ============================================================
# COMPROBAR ESTRUCTURA DE FRESHCHECK-AGENT
# ============================================================

import os

os.chdir('/content/FreshCheck-Agent')

archivos = [
    'src/import_fruitvision.py',
    'src/cache.py',
    'src/features.py',
    'src/baseline_final.py',
    'src/tune.py',
    'src/run_final.py',
    'src/evaluar_final.py',
    'data/manifest.csv',
    'data/splits.csv',
    'experiments/baseline/entorno.json'
]

for archivo in archivos:
    existe = os.path.isfile(archivo)
    print(('OK' if existe else 'FALTA'), '->', archivo)

faltantes = [a for a in archivos if not os.path.isfile(a)]

if faltantes:
    raise FileNotFoundError(
        'Faltan archivos necesarios del proyecto: ' + str(faltantes)
    )

print('\nTODOS LOS ARCHIVOS NECESARIOS ESTÁN PRESENTES.')


OK -> src/import_fruitvision.py
OK -> src/cache.py
OK -> src/features.py
OK -> src/baseline_final.py
OK -> src/tune.py
OK -> src/run_final.py
OK -> src/evaluar_final.py
OK -> data/manifest.csv
OK -> data/splits.csv
OK -> experiments/baseline/entorno.json

TODOS LOS ARCHIVOS NECESARIOS ESTÁN PRESENTES.


## 1. Caché de imágenes y línea base final

In [10]:


# ============================================================
# IMPORTACIÓN DEFINITIVA DE FRUITVISION
# ============================================================

import os
import subprocess
import sys

REPO = '/content/FreshCheck-Agent'

ORIGEN = (
    '/content/drive/MyDrive/FreshCheck/'
    'Original Image/Fruits Original'
)

os.chdir(REPO)

assert os.path.isdir(ORIGEN), 'No se encontró FruitVision'

print('Importando FruitVision...')
print('Origen:', ORIGEN)
print('Destino:', os.path.join(REPO, 'data/raw'))

resultado = subprocess.run(
    [
        sys.executable,
        '-u',
        '-m',
        'src.import_fruitvision',
        '--origen',
        ORIGEN,
        '--excluir-regex',
        'aug'
    ],
    text=True
)

if resultado.returncode != 0:
    raise RuntimeError('Falló la importación de FruitVision.')

print('\nIMPORTACIÓN FINALIZADA.')


Importando FruitVision...
Origen: /content/drive/MyDrive/FreshCheck/Original Image/Fruits Original
Destino: /content/FreshCheck-Agent/data/raw

IMPORTACIÓN FINALIZADA.


In [11]:

# ============================================================
# VERIFICAR IMÁGENES IMPORTADAS
# ============================================================

import os
from pathlib import Path

os.chdir('/content/FreshCheck-Agent')

RAW = Path('data/raw/fruitvision')

assert RAW.is_dir(), 'No se creó data/raw/fruitvision'

print('ESTRUCTURA IMPORTADA:\n')

for clase in ['fresca', 'danada']:
    carpeta = RAW / clase

    if carpeta.is_dir():
        cantidad = sum(1 for p in carpeta.rglob('*') if p.is_file())
        print(clase, '->', cantidad, 'archivos')
    else:
        print(clase, '-> CARPETA NO ENCONTRADA')

# Comprobar imagen que anteriormente faltaba
imagen = RAW / 'danada' / (
    'banano__00eda27f_6184002721938522018.jpg'
)

print('\nImagen que faltaba anteriormente:')
print('ENCONTRADA' if imagen.is_file() else 'NO ENCONTRADA')


ESTRUCTURA IMPORTADA:

fresca -> 3800 archivos
danada -> 3178 archivos

Imagen que faltaba anteriormente:
NO ENCONTRADA


In [16]:

# ============================================================
# IMPORTACIÓN CORRECTA - SIMULACIÓN
# ============================================================

import os
import subprocess
from pathlib import Path

REPO = Path('/content/FreshCheck-Agent')
ORIGEN = Path(
    '/content/drive/MyDrive/FreshCheck/Original Image'
)

os.chdir(REPO)

assert ORIGEN.is_dir(), f'No existe: {ORIGEN}'
assert (ORIGEN / 'Fruits Original').is_dir(), \
    'No se encontró Fruits Original'

print('ORIGEN:', ORIGEN)
print('CARPETA DEL DATASET:', ORIGEN / 'Fruits Original')

resultado = subprocess.run(
    [
        'python', '-m', 'src.import_fruitvision',
        '--origen', str(ORIGEN),
        '--excluir-regex', 'aug',
        '--simular'
    ],
    capture_output=True,
    text=True
)

print(resultado.stdout)
if resultado.stderr:
    print(resultado.stderr)

assert resultado.returncode == 0, 'Falló la simulación'

print('SIMULACIÓN CORRECTA')


ORIGEN: /content/drive/MyDrive/FreshCheck/Original Image
CARPETA DEL DATASET: /content/drive/MyDrive/FreshCheck/Original Image/Fruits Original
Resumen por (carpeta de primer nivel, clase detectada):
  Fruits Original                danada               3178
  Fruits Original                excluida_formalina   3176
  Fruits Original                fresca               3800

Resumen por clase y fruta:
  ('danada', 'banano')                     632
  ('danada', 'mango')                      630
  ('danada', 'manzana')                    630
  ('danada', 'naranja')                    656
  ('danada', 'uva')                        630
  ('fresca', 'banano')                     749
  ('fresca', 'mango')                      763
  ('fresca', 'manzana')                    765
  ('fresca', 'naranja')                    753
  ('fresca', 'uva')                        770
  excluida_formalina                       3176

(simulacion: no se copio nada)

SIMULACIÓN CORRECTA


In [17]:

# ============================================================
# IMPORTACIÓN DEFINITIVA DE FRUITVISION
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

REPO = Path('/content/FreshCheck-Agent')
ORIGEN = Path(
    '/content/drive/MyDrive/FreshCheck/Original Image'
)

DESTINO = REPO / 'data/raw/fruitvision'
RESPALDO = REPO / 'data/raw/fruitvision_backup_incorrecto'

os.chdir(REPO)

assert ORIGEN.is_dir()
assert (ORIGEN / 'Fruits Original').is_dir()

# Evitar sobrescribir una copia de seguridad anterior
assert not RESPALDO.exists(), (
    'Ya existe una copia de seguridad. '
    'Revísala antes de continuar.'
)

# Conservar la importación anterior
if DESTINO.exists():
    shutil.move(str(DESTINO), str(RESPALDO))
    print('Copia de seguridad creada:', RESPALDO)

# Importar utilizando la raíz correcta
resultado = subprocess.run(
    [
        'python', '-m', 'src.import_fruitvision',
        '--origen', str(ORIGEN),
        '--excluir-regex', 'aug'
    ],
    text=True
)

assert resultado.returncode == 0, \
    'La importación ha fallado'

print('\nIMPORTACIÓN TERMINADA')


Copia de seguridad creada: /content/FreshCheck-Agent/data/raw/fruitvision_backup_incorrecto

IMPORTACIÓN TERMINADA


In [18]:

# ============================================================
# VERIFICACIÓN DEFINITIVA
# ============================================================

import os
import pandas as pd
from pathlib import Path

os.chdir('/content/FreshCheck-Agent')

m = pd.read_csv('data/manifest.csv')

faltan = [
    identificador
    for identificador in m['id']
    if not (Path('data/raw') / str(identificador)).is_file()
]

print('Imágenes en manifiesto:', len(m))
print('Imágenes faltantes:', len(faltan))

if faltan:
    print('\nPRIMERAS IMÁGENES FALTANTES:')
    print(faltan[:10])
else:
    print('\nTODAS LAS IMÁGENES COINCIDEN CON EL MANIFIESTO')

assert len(faltan) == 0, 'Todavía hay imágenes faltantes'

print('\nFRESHCHECK: DATASET VALIDADO')


Imágenes en manifiesto: 6978
Imágenes faltantes: 0

TODAS LAS IMÁGENES COINCIDEN CON EL MANIFIESTO

FRESHCHECK: DATASET VALIDADO


In [ ]:
!python -m src.cache
!python -m src.features
!python -m src.baseline_final

In [19]:

# ============================================================
# 1. CACHÉ DE IMÁGENES Y LÍNEA BASE FINAL
# ============================================================

import os
import subprocess
import sys

os.chdir('/content/FreshCheck-Agent')

for modulo in [
    'src.cache',
    'src.features',
    'src.baseline_final'
]:
    print('\nEJECUTANDO:', modulo)

    subprocess.run(
        [sys.executable, '-u', '-m', modulo],
        check=True
    )

print('\nETAPA 1 COMPLETADA.')



EJECUTANDO: src.cache

EJECUTANDO: src.features

EJECUTANDO: src.baseline_final

ETAPA 1 COMPLETADA.


## 2. Búsqueda de hiperparámetros (solo validación; la prueba no se toca)

In [20]:

# ============================================================
# 2. BÚSQUEDA DE HIPERPARÁMETROS
# SOLO VALIDACIÓN: LA PRUEBA NO SE TOCA
# ============================================================

import os
import subprocess
import sys

os.chdir('/content/FreshCheck-Agent')

RAPIDO = True

ARQS = (
    'efficientnet_b0'
    if RAPIDO
    else 'resnet50 efficientnet_b0'
)

EPOCAS = 4 if RAPIDO else 6

print('Modo rápido:', RAPIDO)
print('Arquitecturas:', ARQS)
print('Épocas:', EPOCAS)

subprocess.run(
    [
        sys.executable,
        '-m', 'src.tune',
        '--arqs', ARQS,
        '--epocas', str(EPOCAS)
    ],
    check=True
)

print('\nBÚSQUEDA DE HIPERPARÁMETROS COMPLETADA.')


Modo rápido: True
Arquitecturas: efficientnet_b0
Épocas: 4

BÚSQUEDA DE HIPERPARÁMETROS COMPLETADA.


## 3. Corridas finales (3 semillas, evaluación única en prueba) y fruta no vista

In [21]:

# ============================================================
# 3. CORRIDAS FINALES
# 3 SEMILLAS Y EVALUACIÓN FINAL
# ============================================================

import os
import subprocess
import sys

os.chdir('/content/FreshCheck-Agent')

# Entrenamiento final
subprocess.run(
    [
        sys.executable,
        '-m', 'src.run_final',
        '--arqs', ARQS
    ],
    check=True
)

print('\nENTRENAMIENTO FINAL COMPLETADO.')

# Evaluación final
subprocess.run(
    [
        sys.executable,
        '-m', 'src.evaluar_final'
    ],
    check=True
)

print('\nEVALUACIÓN FINAL COMPLETADA.')



ENTRENAMIENTO FINAL COMPLETADO.

EVALUACIÓN FINAL COMPLETADA.


In [22]:

# ============================================================
# 4. GUARDAR LOS RESULTADOS DEL HITO 3
# ============================================================

import os
import zipfile
import shutil
from pathlib import Path
from google.colab import files

os.chdir('/content/FreshCheck-Agent')

# Archivos y carpetas que se guardarán
ELEMENTOS = [
    Path('experiments/final'),
    Path('models_v2'),
    Path('docs/analisis_errores.csv')
]

# Comprobar resultados
faltantes = [
    str(p) for p in ELEMENTOS
    if not p.exists()
]

if faltantes:
    raise FileNotFoundError(
        'No se encontraron todos los resultados:\n'
        + '\n'.join(faltantes)
    )

# Crear ZIP
SALIDA = Path('resultados_hito3.zip')

with zipfile.ZipFile(
    SALIDA,
    'w',
    compression=zipfile.ZIP_DEFLATED
) as z:
    for elemento in ELEMENTOS:

        if elemento.is_file():
            z.write(elemento, elemento.as_posix())

        elif elemento.is_dir():
            for archivo in elemento.rglob('*'):
                if archivo.is_file():
                    z.write(archivo, archivo.as_posix())

print('ZIP creado correctamente.')
print('Tamaño:', round(SALIDA.stat().st_size / (1024**2), 2), 'MB')

# Copiar a Google Drive
DESTINO = Path(
    '/content/drive/MyDrive/FreshCheck/resultados_hito3.zip'
)

shutil.copy2(SALIDA, DESTINO)

print('\nResultados guardados en:')
print(DESTINO)

# Descargar resultados
files.download(str(SALIDA))

print('\nHITO 3 COMPLETADO.')


ZIP creado correctamente.
Tamaño: 8.39 MB

Resultados guardados en:
/content/drive/MyDrive/FreshCheck/resultados_hito3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


HITO 3 COMPLETADO.
